# Apple Silicon CPU/GPU selection

Sweep exact statevector widths around MettleQ's measured GPU crossover and verify parity at each width.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the scalable workload

Qiskit's reference statevector already uses the Apple CPU. MettleQ adds an MLX/Metal GPU path and selects it only after a measured crossover.

In [2]:
import statistics

widths = [12, 14, 16, 18, 20]
rows = []

## 2. Run the same workload through both simulators

For each width we warm up both paths twice, take five measured runs, compare the complete state, record MettleQ's selected device, and print the result as a per-width table. A ratio above 1.0 means MettleQ was faster; below 1.0 means the SDK reference was faster.

In [3]:
for width in widths:
    circuit = QuantumCircuit(width)
    for layer in range(3):
        for wire in range(width):
            circuit.ry(0.01 * (layer + 1) * (wire + 1), wire)
        for wire in range(layer % 2, width - 1, 2):
            circuit.cx(wire, wire + 1)
    reference, reference_ms, _ = benchmark(
        lambda c=circuit: np.asarray(Statevector.from_instruction(c).data),
        warmups=2,
        repeats=5,
    )
    backend = MettleQBackend(method="statevector", device="auto")
    compiled = transpile(circuit, backend, optimization_level=1)
    def run_mettleq(c=compiled, b=backend):
        return np.asarray(b.run(c, shots=1, return_statevector=True).result().data(0)["statevector"])
    candidate, mettleq_ms, _ = benchmark(run_mettleq, warmups=2, repeats=5)
    method, device = qiskit_selection(backend)
    rows.append({
        "width": width,
        "reference_ms": reference_ms,
        "mettleq_ms": mettleq_ms,
        "error": phase_aligned_statevector_error(reference, candidate),
        "method": method,
        "device": device,
    })

print_scaling_table(rows)
largest_width_speedup = rows[-1]["reference_ms"] / rows[-1]["mettleq_ms"]
passed = (
    all(row["error"] <= 3e-6 for row in rows)
    and rows[-1]["device"] == "gpu"
    and largest_width_speedup >= 1.5
)

qubits | reference ms | MettleQ ms | ref/MettleQ | path | max error
------ | ------------ | ---------- | ----------- | ---- | ---------
    12 |        2.386 |      2.865 |      0.833x | statevector/cpu | 2.26e-07
    14 |        5.038 |      5.772 |      0.873x | statevector/cpu | 2.27e-07
    16 |       10.678 |      3.460 |      3.086x | statevector/gpu | 8.76e-08
    18 |       63.265 |      5.704 |     11.091x | statevector/gpu | 1.21e-07
    20 |      441.510 |      5.888 |     74.984x | statevector/gpu | 8.31e-08


## 3. Enforce correctness and publish the evidence

Every width is compared to Qiskit's full statevector before its timing is interpreted.

In [4]:
tutorial_result = emit_result(
    notebook="qiskit/15_apple_gpu_scaling.ipynb",
    framework="qiskit",
    reference_ms=statistics.median(row["reference_ms"] for row in rows),
    mettleq_ms=statistics.median(row["mettleq_ms"] for row in rows),
    check="per-width statevector atol=3e-6, policy-selected GPU, and 20q speedup >=1.5x",
    passed=passed,
    exact_match=all(row["error"] == 0.0 for row in rows),
    selected_method=rows[-1]["method"],
    selected_device=rows[-1]["device"],
    metrics={"widths": rows},
    notes="The aggregate medians summarize different widths. The printed per-width table is the performance evidence; ratios above 1 mean MettleQ was faster.",
)


Comparison summary
------------------
Correctness contract: PASS — per-width statevector atol=3e-6, policy-selected GPU, and 20q speedup >=1.5x
SDK reference median: 10.678 ms
MettleQ median:       5.704 ms
Timing interpretation: MettleQ was 1.872x faster in this run.
MettleQ selected: statevector / gpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)
Note: The aggregate medians summarize different widths. The printed per-width table is the performance evidence; ratios above 1 mean MettleQ was faster.

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "per-width statevector atol=3e-6, policy-selected GPU, and 20q speedup >=1.5x", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"widths": [{"device": "cpu", "error": 2.26429405669748e-07, "method": "statevector", "mettleq_ms": 2.864792011678219, "reference_ms": 2.3859579814597964, "width": 12}, {"device": "cpu", "error": 2.2651331799128371e-07, "met

## What should you conclude?

This is the performance-decision notebook: below crossover use the reference or CPU path; above crossover the same circuit can amortize GPU dispatch.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.